```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 1: DRL on Flat Terrain

**Goal:** train a policy that makes the Go2 follow velocity commands in any direction on flat ground, then test one design choice of your own in a controlled experiment.

| Section | What you do |
| --- | --- |
| 1.1 | Build the environment, i.e. the MDP the policy learns in |
| 1.2 | Configure the policy network and the PPO algorithm |
| 1.3 | Train a baseline and monitor the training |
| 1.4 | Test the policy and collect evaluation data |
| 1.5 | Run a controlled experiment against your baseline |

This notebook **produces data**: trained policies, training logs and evaluation rollouts, all saved in `logs/`. 
It **requires a GPU for training**, evaluation and visualization can possibly be done on CPU but at a much slower rate.
You analyse the produced data in `logs/` in `02_flat_analysis.ipynb`, which is lightweight and runs entirely on CPU.
Take this into account when dividing work within your group!

**Time budget.** A training run takes roughly 10 to 20 minutes of shared GPU time. 
Check your environment in the viewer (1.1h) before training, and change one thing at a time between runs so you know what caused a difference.

If you are using gnoto, the GPUs might be busy during the practicals and **close to the deadline**.
Make sure to plan your training runs accordingly!

In [1]:
%load_ext autoreload
%autoreload 2

import math
import copy
import lr_project # Before mjlab import

from mjlab.envs import ManagerBasedRlEnv, ManagerBasedRlEnvCfg
from mjlab.rl import RslRlModelCfg, RslRlPpoAlgorithmCfg, RslRlOnPolicyRunnerCfg

from mjlab.managers.action_manager import ActionTermCfg
from mjlab.managers.command_manager import CommandTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.observation_manager import ObservationTermCfg, ObservationGroupCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg

from lr_project import mdp, scene, play, train, evaluate, registry

# Added by us
import torch
from mjlab.entity import Entity
from mjlab.tasks.velocity import mdp as vel_mdp
from mjlab.managers.scene_entity_config import SceneEntityCfg

In [2]:
# Custom reward terms (ours): defined here, not in src/, because only the notebooks are submitted.


def feet_clearance_flat(
    env: ManagerBasedRlEnv,
    target_height: float,
    asset_cfg: SceneEntityCfg = SceneEntityCfg("robot"),
) -> torch.Tensor:
    """|foot height - target| weighted by horizontal foot speed: only swinging feet pay."""
    asset: Entity = env.scene[asset_cfg.name]
    height = asset.data.site_pos_w[:, asset_cfg.site_ids, 2] - 0.022  # foot radius; flat plane only
    speed = torch.norm(asset.data.site_lin_vel_w[:, asset_cfg.site_ids, :2], dim=-1)
    return torch.sum(torch.abs(height - target_height) * speed, dim=1)


def stand_still_joint_deviation(
    env: ManagerBasedRlEnv,
    command_name: str,
    command_threshold: float = 0.1,
    asset_cfg: SceneEntityCfg = SceneEntityCfg("robot"),
) -> torch.Tensor:
    """sum |q - q_default| (rad), only when the command is ~zero: holds the standing pose instead of crouching."""
    asset: Entity = env.scene[asset_cfg.name]
    command = env.command_manager.get_command(command_name)
    assert command is not None, f"Command '{command_name}' not found."
    deviation = torch.sum(
        torch.abs(asset.data.joint_pos[:, asset_cfg.joint_ids] - asset.data.default_joint_pos[:, asset_cfg.joint_ids]),
        dim=1,
    )
    # Same gate as feet_air_time: linear + angular command magnitude.
    standing = (torch.norm(command[:, :2], dim=1) + torch.abs(command[:, 2])) < command_threshold
    return deviation * standing.float()

## 1.1 Building the environment configuration

In mjlab, an environment is described by a configuration class, the `ManagerBasedRLEnvCfg`.
It is built from several parts, each defining one piece of the Markov Decision Process (MDP) which you will define in the sections below:


| MDP element | Config part | Section |
| --- | --- | --- |
| Task (what the robot should do) | commands | 1.1a |
| Observation $o_t$ | observations | 1.1b |
| Action $a_t$ | actions | 1.1c |
| Reward $r_t$ | rewards | 1.1d |
| End of an episode | terminations | 1.1e |
| Initial state distribution | events | 1.1f |
| Transition dynamics $P$ | MuJoCo physics (provided by `scene`) | 1.1g |

Each `make_*` function you will complete below returns a dictionary `{name: term config}`.
A term is a **function** (`func`) plus its optional **parameters** (`params`).
We provide you with ready-made functions in `src/lr_project/mdp/*.py` (skim through it!), and mjlab provides more in `mjlab.envs.mdp`.
You can also write your own functions in a cell and use it!

For more information on mjlab's syntax and available functions, consult the [official documentation](https://mujocolab.github.io/mjlab/main/index.html).


Every `make_*` function receives a `play` argument.
It is false for training (4096 parallel robots) and true for visualization and evaluation (1 robot, debug rendering of the commands).
Use it if a term should behave differently in the two cases.

**Advice:** it can be a bit daunting to define everything from scratch, where exactly do you start?
A helpful source of inspiration is taking a look at existing literature, for example the articles from the student presentations.
A starting point we recommend is the following paper by Rudin et al., which is a solid baseline for training quadruped locomotion policies using GPUs:

> Rudin, N., Hoeller, D., Reist, P. &amp; Hutter, M.. (2022). Learning to Walk in Minutes Using Massively Parallel Deep Reinforcement Learning. <i>Proceedings`of the 5th Conference on Robot Learning</i>, in <i>Proceedings of Machine Learning Research</i> 164:91-100 Available from https://proceedings.mlr.press/v164/rudin22a.html.

Note that this article also does challenging terrain, so implementing it exactly would be overkill for this policy that only needs to walk on flat ground.

### 1.1a Commands

The command defines the task: 
a desired base velocity $(v_x, v_y, \omega_z)$ in the robot's frame (forward and lateral speed in m/s, yaw rate in rad/s). 
Each robot samples a command uniformly from the ranges below and samples a new one every `resampling_time_range` seconds. 
The policy sees the command as an observation (1.1b) and is rewarded for tracking it (1.1d).

Choose ranges the Go2 can realistically reach. 
Things to think about:
- Why resample the command during an episode instead of once per episode?
- What happens if no robot is ever commanded to stand still (`rel_standing_envs`)?
- `rel_forward_envs` is the fraction of robots that only walk forward. Why could that help or hurt?

**Note:** there is a bug in mjlab's viewer when the command ranges are identical (it crashes), make sure to define them to realistic ranges before running the viewer.

In [3]:
def make_commands(play: bool) -> dict[str, CommandTermCfg]:
    return {
        "joystick": mdp.UniformVelocityCommandCfg(
            resampling_time_range=(5, 7), # TODO
            rel_standing_envs=0.10, # TODO
            rel_forward_envs=0.2, # TODO
            ranges=mdp.UniformVelocityCommandCfg.Ranges(
                lin_vel_x=(-1.0, 1.0), # TODO
                lin_vel_y=(-0.5, 0.5), # Rudin (-0.1, 0.1)
                ang_vel_z=(-0.5, 0.5), # TODO
            ),
            entity_name="robot",
            debug_vis=play, # Do not render command arrows during training
        )
    }

### 1.1b Observations

Observations are what the networks see at every environment step. 
There are two groups:

- **actor**: the input of the policy. 
The policy will run on the robot, so think about what a real Go2 can measure or know before adding the observation.
- **critic**: the input of the value function. 
The critic is only used during training, so it may also receive *privileged* information that is easy to get in simulation but hard to measure on hardware (this is called an asymmetric actor-critic).

Terms are concatenated in the order you list them. 
Available functions are in `src/lr_project/mdp/observations.py`.

Things to think about:
What does the policy need to know at inference time?
Can the real robot make the measurements for the policy?
Can the critic infer the reward from its observations alone?

In [4]:
def make_observations(play: bool) -> dict[str, ObservationGroupCfg]:
    actor = {

        "command": ObservationTermCfg(
            func=mdp.generated_command,
            params={"command_name": "joystick"},
        ),

        "joint_pos": ObservationTermCfg(
            func=mdp.joint_positions,
        ),

        "joint_vel": ObservationTermCfg(
            func=mdp.joint_velocities,
        ),

        "base_angular_vel": ObservationTermCfg(
            func=mdp.base_ang_vel_b,
        ),

        "gravity": ObservationTermCfg(
            func=mdp.projected_gravity_b,
        ),

        "last_action": ObservationTermCfg(
            func=mdp.last_action,
        ),

        "last_last_action": ObservationTermCfg(
            func=mdp.last_last_action,
        ),  

        "last_last_last_action": ObservationTermCfg(
            func=mdp.last_last_last_action,
        ),
    }

    critic = {
        **actor,  # the critic sees everything the actor sees...
        # ...plus privileged terms (only available in simulation)

        "base_lin_vel": ObservationTermCfg(
            func=mdp.base_lin_vel_b,
        ),
        "foot_contact": ObservationTermCfg(
            func=mdp.foot_contact,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
        "foot_air_time": ObservationTermCfg(
            func=mdp.foot_air_time,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
        "foot_contact_forces": ObservationTermCfg(
            func=mdp.foot_contact_forces,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
        "joint_torques": ObservationTermCfg(
            func=mdp.joint_torques,
        ),
#
        "foot_vel": ObservationTermCfg(
            func=mdp.builtin_sensors,
            params={"sensor_names": scene.FOOT_VEL_SENSORS},
        ),
    }

    return {
        "actor": ObservationGroupCfg(terms=actor),
        "critic": ObservationGroupCfg(terms=critic),
    }

### 1.1c Actions

The policy outputs 12 numbers, one per joint. 
With `JointPositionActionCfg`, each output is turned into a joint position target:

$$q^{target} = q^{default} + \text{scale} \cdot a_t \quad (\text{with } \texttt{use\_default\_offset=True})$$

A PD controller in the robot model then computes the motor torques that move the joint towards this target. 
The policy runs at 50 Hz, the PD controller and physics at 200 Hz.

Things to think about: Why predict offsets from the standing pose instead of absolute angles? What happens if `scale` is too small or too large?

While this is a common action space for quadrupeds, there are many other ways to translate policy actions into motor commands, and these can have a huge effect on your policy's performance!
For instance, one can use the policy outputs to modulate a CPG, as done previously in our lab:

> Bellegarda, G., & Ijspeert, A. (2022). CPG-RL: Learning central pattern generators for quadruped locomotion. IEEE Robotics and Automation Letters, 7(4), 12547-12554.

If interested in the topic, check out this paper for more action spaces and their effects on learning:

> Peng, X. B., & Van De Panne, M. (2017, July). Learning locomotion skills using deeprl: Does the choice of action space matter?. In Proceedings of the ACM SIGGRAPH/Eurographics Symposium on Computer Animation (pp. 1-13).

In [5]:
def make_actions(play: bool) -> dict[str, ActionTermCfg]:
    return {
        "joint_targets": mdp.JointPositionActionCfg(
            entity_name="robot", # Control the Go2 robot in the scene.
            actuator_names=(".*"), # Select all 12 leg motors: 3 per leg.
            use_default_offset=True,
            scale=0.25,
        ),
    }

### 1.1d Rewards

At every step the robot receives $r_t = \sum_i w_i \, f_i(s_t, a_t) \, \Delta t$: 
each term $f_i$ is multiplied by its weight $w_i$ (positive = reward, negative = penalty) and by the policy time step $\Delta t$.
Available functions are in `src/lr_project/mdp/rewards.py`:

- **Task**: `track_linear_velocity`, `track_angular_velocity` reward $\exp(-\text{error}^2 / \text{std}^2)$, so `std` sets how tolerant the tracking is (plot this function for better intuition!)
- **Penalties** (regularization): body velocities, joint velocities, accelerations, torques and power, action rate, contacts, ...
- **Shaping**: `feet_air_time` encourages stepping over jittering.

Terms that take a `sensor_name` need a sensor defined in the scene (the feet contact sensor is called `scene.FEET_CONTACT_SENSOR`).
Check `src/lr_project/scene.py` for more sensor names.

**How to tune:** 
start with the tracking terms and a few (minimal) penalties, then add or change **one term at a time**. 
Watch each term's curve (`Episode_Reward/<name>`) in TensorBoard (see 1.3a). 
If the robot finds an unexpected way to collect reward (e.g. vibrating in place), ask which term allows it. 

Keep a short log of what you tried!
You will explain your choices in notebook 02 and we will ask about them during the oral exam.

In [6]:
def make_rewards(play: bool) -> dict[str, RewardTermCfg]:


    feet = SceneEntityCfg("robot", site_names=scene.FOOT_SITES) # Added by us

    return {

        # Earn points for moving at the requested forward/sideways speed.
        "track_linear_velocity": RewardTermCfg(
            func=mdp.track_linear_velocity,
            weight=1.0,
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),

        # Earn points for turning at the requested rate.
        "track_angular_velocity": RewardTermCfg(
            func=mdp.track_angular_velocity,
            weight=1.0,
            params={"command_name": "joystick", "std": math.sqrt(0.5)},
        ),

        # Lose points for large motor torques, also when the joints are not moving.
        "joint_torques": RewardTermCfg(
            func=mdp.joint_torques_l2,
            weight=-0.0003,  # replace with -0.05 * 1.968 / typical value
        ),

        # Lose points for the mechanical power delivered by the motors (sum |torque * joint velocity|, W).
        "joint_power": RewardTermCfg(
            func=mdp.joint_power,
            weight=-0.001,
        ),

        # Lose points for leaning, even if the body is not rocking.
        "body_tilt": RewardTermCfg(
            func=mdp.flat_orientation_l2,
            weight=-1.5,
        ),

        # Lose points for jerking the virtual joysticks between policy steps.
        "action_rate": RewardTermCfg(
            func=mdp.action_rate_l2,
            weight=-0.03, # Rudin is -0.01 
        ),

        # Lose points for jerking the virtual joysticks between policy steps.
        "action_rate": RewardTermCfg(
            func=mdp.action_rate_l2,
            weight=-0.03, # Rudin is -0.01 
        ),

        "foot_clearance": RewardTermCfg(
            func=feet_clearance_flat, weight=-1.0,
            params={"target_height": 0.08, "asset_cfg": feet},
        ),
        
        "foot_slip": RewardTermCfg(
            func=vel_mdp.feet_slip, weight=-0.3,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR, "command_name": "joystick", "asset_cfg": feet},
        ),

        # Lose points for leaving the default standing pose when the command is zero.
        "stand_still": RewardTermCfg(
            func=stand_still_joint_deviation,
            weight=-0.5,
            params={"command_name": "joystick"},
        ),

        # Lose points for the mechanical power of the motors (sum |torque * joint velocity|, W),
        # scaled by ref_speed / commanded speed (adaptive energy weight).
        #"joint_power_adaptive": RewardTermCfg(
        #    func=mdp.joint_power_adaptive,
        #    weight=-0.001,  # replace with -0.05 * 1.968 / typical value
        #    params={"command_name": "joystick", "ref_speed": 1.0},
        #),
        


        # Earn a small bonus for lifting feet while movement is requested.
        #"feet_air_time": RewardTermCfg(
        #    func=mdp._reward_feet_air_time,
        #    weight=0.1,
        #    params={
        #        "sensor_name": scene.FEET_CONTACT_SENSOR,
        #        "command_name": "joystick",
        #    },
        #),

        #"feet_air_time": RewardTermCfg(
        #    func=mdp.feet_air_time,
        #    weight=0.1,
        #    params={
        #        "sensor_name": scene.FEET_CONTACT_SENSOR,
        #        "command_name": "joystick",
        #        "threshold_max": 0.1,   # long flights (hops) no longer count
        #    },
        #),

        # Lose points for bouncing up/down along the body's z-axis.
        #"body_linear_velocity_penalty": RewardTermCfg(
        #    func=mdp.body_linear_velocity_penalty,
        #    weight=-0.5, #Rudin is -2.0
        #),

        # Lose points for rocking sideways or pitching back/forth; yaw turning is allowed.
        #"body_angular_velocity_penalty": RewardTermCfg(
        #    func=mdp.body_angular_velocity_penalty,
        #    weight=-0.05,
        #),


        # Lose points for leg contacts.
        #"leg_contact_penalty": RewardTermCfg(
        #    func=mdp.contacts_penalty,
        #    weight=-0.1,
        #    params={"sensor_name": scene.LEG_CONTACT_SENSOR},
        #),

        # Lose points for body contacts.
        #"base_contact_penalty": RewardTermCfg(
        #    func=mdp.contacts_penalty,
        #    weight=-0.3,
        #    params={"sensor_name": scene.BASE_CONTACT_SENSOR},
        #)

    }


### 1.1e Terminations

An episode ends when any termination term returns `True` for a robot, and that robot is reset (1.1f).
There are two kinds:

- **Time-outs** (`time_out=True`): 
the episode reached its time limit. 
This is not a failure: PPO keeps estimating the value of the last state, as if the robot could continue (why is this necessary?)
- **Failures** (`time_out=False`): 
the robot fell. 
The robot loses all future rewards, which acts as an implicit penalty, and no more samples are wasted on a robot lying on the ground.

Things to think about: 
What should count as a fall? 
What happens if the failure condition is too strict or too loose?

In [7]:
def make_terminations(play: bool) -> dict[str, TerminationTermCfg]:
    return {
        # Example
        "time_out": TerminationTermCfg(
            func=mdp.time_out,
            time_out=True, # TODO: is this a failure or not?
        ),
        # TODO: add at least one failure termination that detects a fall

        # The body has tipped more than 60 degrees away from upright.
        "bad_orientation": TerminationTermCfg(
            func=mdp.bad_orientation,
            params={"limit_angle": math.radians(60)},
            time_out=False,
        ),

        # The base has dropped below 15 cm: treat this as a collapse.
        "low_height": TerminationTermCfg(
            func=mdp.root_height_below_minimum,
            params={"minimum_height": 0.15},
            time_out=False,
        ),

        # End the attempt if the robot's body touches the ground.
        "base_contact": TerminationTermCfg(
            func=mdp.illegal_contact,
            params={"sensor_name": scene.BASE_CONTACT_SENSOR},
            time_out=False,
        ),

    }

### 1.1f Events

Events are functions that modify the simulation at given moments, set by `mode`:

- `"startup"`: once, when the environment is created;
- `"reset"`: every time a robot starts a new episode;
- `"interval"`: periodically during an episode (e.g. pushes).

Here, reset events define the **initial state distribution**: where and how robots start. 
`pose_range` gives ranges for the base position (m, relative to its spawn point) and orientation (rad). 
`reset_joints_by_offset` can also randomize the initial joint angles.

Things to think about:
Why do the initial poses need to be randomized?
Is it as important on flat terrain than on a challenging terrain scenario?

In [8]:
def make_events(play: bool) -> dict[str, EventTermCfg]:
    return {
        "reset_base": EventTermCfg(
            func=mdp.reset_root_pose_uniform,
            mode="reset",
            params={
                "pose_range": {
                    "x": (-0.25, 0.25),
                    "y": (-0.25, 0.25),
                    "z": (0.04, 0.06),
                    "roll": (-0.05, 0.05),
                    "pitch": (-0.05, 0.05),
                    "yaw": (-math.pi, math.pi),
                },
            },
        ),

        "reset_joints": EventTermCfg(
            func=mdp.reset_joints_by_offset,
            mode="reset",
            params={
                "position_range": (-0.05, 0.05),
                "velocity_range": (0.0, 0.0),
            },
        ),
    }

### 1.1g Assembly

All parts come together here, with the scene (robot, flat terrain, feet contact sensor) and the simulator settings. 
The physics runs at 200 Hz (`timestep_s=0.005`) and the policy acts every 4 physics steps (`decimation=4`), i.e. at 50 Hz. 
An episode lasts 20 s = 1000 policy steps. 
You normally do not need to change this cell.

In [9]:
def make_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    return ManagerBasedRlEnvCfg(
        scene=scene.make_scene_cfg(
            num_envs=1 if play else 4096,
            terrain="flat",
            robot_entity_name="robot",
        ),
        sim=scene.make_sim_cfg(timestep_s=0.005),
        actions=make_actions(play),
        commands=make_commands(play),
        events=make_events(play),
        observations=make_observations(play),
        rewards=make_rewards(play),
        terminations=make_terminations(play),
        decimation=4,
        episode_length_s=20.0,
        seed=seed,
    )

### 1.1h Visualize your environment

Run your environment with a `"random"` or `"zero"` policy before training. 
Check that robots start where you expect, that the command arrows look right, and that a fallen robot is reset. 
Problems found here cost seconds while problems found after training cost a training run!

Stop the viewer with the **interrupt** button (■) before running the next cell.

In [10]:
env_cfg = make_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

Setting seed: 42
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.2
   Devices:
     "cpu"      : "CPU"
     "cuda:0"   : "NVIDIA GeForce RTX 5060 Laptop GPU" (8 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/carolina-araujo/.cache/warp/1.18.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 4.64 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 1.88 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_9914afce 9914afc load on device 'cuda:0' took 1.96 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 0.74 ms  (cached)
Module _friction_dof__locals__kernel_35a89db9 35a89db load on device 'cuda:0' took 1.10 ms  (cached)
Module _limit_slide_hinge__locals__kernel_6395e840 6395e84 load on device 'cuda:0' took 0.94 ms  (cached)
Module _efc_contact_init__locals__kernel_81f4db55 81f4db5 load on device 'cuda:0' took 0.84 ms  (cached)
Module _efc_contact_j

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: http://localhost:8080


KeyboardInterrupt: 

## 1.2 Building the training configuration

The policy is trained with **PPO** (Proximal Policy Optimization), an actor-critic algorithm from the `rsl_rl` library. 
Each iteration has two phases:

1. **Collect:** all 4096 robots run the current policy for `num_steps_per_env` steps.
2. **Update:** the actor and critic networks are improved using the collected data.

The values below are a reasonable starting point. 
**Change them only with a reason** (i.e., after having a working baseline), and one at a time.

### 1.2a Actor model

The actor is the policy: a neural network (MLP) mapping the actor observations to the mean of a Gaussian distribution over actions. 
During training, actions are sampled from this distribution to explore: `init_std` is the initial exploration noise, and the standard deviation is learned. 
`obs_normalization` rescales each observation using running statistics (why is this needed?).

In [10]:
def make_actor_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
        distribution_cfg={
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
            "std_type": "scalar",
        },
    )

### 1.2b Critic model

The critic estimates the value $V(s)$: 
the expected discounted sum of future rewards from a state. 
PPO uses it to judge whether an action turned out better or worse than expected (the *advantage*). 
It takes the critic observations as input and is not used once training is finished.

In [11]:
def make_critic_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
    )

### 1.2c PPO hyperparameters

| Parameter | Meaning |
| --- | --- |
| `gamma` | Discount factor $\gamma$: how much future rewards count. $\gamma = 0.995$ at 50 Hz looks about 4 s ahead. |
| `lam` | GAE $\lambda$: trade-off between bias and variance of the advantage estimate. |
| `clip_param` | Limits how much the policy can change in one update (stabilizes training). |
| `entropy_coef` | Bonus for keeping the action distribution wide (exploration). |
| `num_learning_epochs`, `num_mini_batches` | How many passes over the collected data, split into how many batches. |
| `learning_rate`, `schedule`, `desired_kl` | Learning rate for fitting the neural networks, `"adaptive"` lowers it if the policy changes more than `desired_kl` per update. |
| `value_loss_coef`, `use_clipped_value_loss` | Weight and clipping of the critic loss. |
| `max_grad_norm` | Gradient clipping, for stability. |


In [18]:
def make_ppo_cfg() -> RslRlPpoAlgorithmCfg:
    return RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.01,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",
        gamma=0.995,
        lam=0.97,
        desired_kl=0.01,
        max_grad_norm=1.0,
    )

### 1.2d Assembly

`make_rl_cfg` combines the actor-criticm models with the PPO hyperparameters, and sets the size of training: 
each iteration collects 4096 robots × 24 steps ≈ 98k transitions, for `max_iterations` iterations. 
A checkpoint is saved every `save_interval` iterations and at the end. 
The `seed` makes a run reproducible: train the same configuration with different seeds to see how much results vary by chance!

**Do not change the `max_iterations` argument!**
For flat terrain locomotion, your policies should comfortably converge within 500 iterations (and keeps training within acceptable time frames for shared GPU usage).
If it doesn't, tune your MDP further.

In [12]:
def make_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        seed=seed,
        num_steps_per_env=24,
        max_iterations=500, # DO NOT INCREASE
        save_interval=500,
        actor=make_actor_cfg(),
        critic=make_critic_cfg(),
        algorithm=make_ppo_cfg(),
    )

## 1.3 Training

In this section you will run and monitor your training using the previously defined MDP and RL configurations.

### 1.3a Monitoring your training with Tensorboard

Start TensorBoard **before** training, then open the printed link in a new tab. 
Refresh the page to update the curves while training runs.
Useful curves to monitor and compare across trainings:

- `Train/mean_reward` and `Train/mean_episode_length` (1000 steps = no robot falls within 20 s);
- `Episode_Reward/<term>`: the contribution of each reward term, to see what the policy is optimizing;
- `Loss/*` and the policy's action standard deviation, to see whether learning is stable and exploration shrinks.

In [14]:
train.tensorboard()

Tensorboard server running! Monitor your training here: http://localhost:48353/


### 1.3b Running your training

Training saves everything in `logs/<experiment>/<date>_<variant>/`. 
Running the cell again starts a **new** run, previous runs are kept.
Make sure to back up your `logs` as producing the data and checkpoints is computationally expensive.

- Use `experiment="flat"` for all runs of this task, and `variant` to name what the run tests (initially your baseline)
- To train the same configuration with another seed, pass `seed=...` to both `make_env_cfg` and `make_rl_cfg` and keep the same variant name.

In [ ]:
train.train(
    env_cfg=make_env_cfg(),
    rl_cfg=make_rl_cfg(),
    experiment="flat",
    variant="baseline",
)

[train] run dir: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-20_13-35-09_baseline
[train] device: cuda:0, envs: 4096, seed: 42
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms

PosixPath('/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-20_13-35-09_baseline')

### 1.3c Loading your latest training directory

The registry finds runs by experiment and variant, so this cell also works after a kernel restart: 
you never need to retrain to continue and can skip the training cell. 
`runs(...)` returns the runs sorted by creation time, and `[-1]` takes the latest.

In [13]:
run = registry.runs(experiment="flat", variant="baseline")[-1]
print(f"Found checkpoint at: {run.checkpoint}")

Found checkpoint at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-19_18-46-30_baseline/model_499.pt


## 1.4 Evaluation

### 1.4a Testing your policy 

Watch your trained policy in the viewer. 
Does it follow the commands? 
Does the gait look natural, or does it shuffle, hop or vibrate? 
Is it stable when the command changes? 
What you see here guides your next reward changes!
Stop the viewer (■) when done.

In [ ]:
play.interactive(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    policy=run.checkpoint,
)

Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 8      |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----------+------------------------+
| Index | Name     |          

╭────── viser (listening *:8082) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8082   │
│   Websocket │ ws://localhost:8082     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: http://localhost:8082


(viser) Connection opened (0, 1 total), 248 persistent messages


Ctrl+C received. Shutting down viewer...


(viser) Connection closed (0, 0 total)

(viser) Server stopped

Viewer closed!


/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


### 1.4b Collecting rollout data for analysis

`collect_rollouts` runs the policy on a fixed grid of commands, with `robots_per_command` robots per command for `duration_s` seconds, and records everything (base motion, joint states, torques, foot contacts and ground reaction forces). 
The data is saved as a `Rollout` file in the run directory, under `eval/<name>_<checkpoint>.npz`, and is what you analyse in notebook 02.
For more information on what exactly gets recorded, check out `src/lr_project/rollout.py`.

- Running the cell again **loads** the saved file instead of simulating again, unless the settings changed (or `overwrite=True`).
- Use the **same commands and name** for every run you want to compare, so the comparison is fair.

In [14]:
# Fixed commands to evaluate: all combinations of the values below
commands = evaluate.command_grid(
    # TODO: what commands are you interested in evaluating?

    vx=(-1.0, -0.5, 0.5, 1.0), # m/s
    vy=(-0.3, 0, 0.3), # m/s
    wz=(-0.3, 0, 0.3), # rad/s
)

# Name of the saved evaluation, reuse it for every run you compare
eval_name = "command_grid"

evaluate.collect_rollouts(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    checkpoint=run.checkpoint,
    commands=commands,
    name=eval_name,
    robots_per_command=10, # robots per unique command combination
    duration_s=10.0,
    seed=42,
)

print(f"Saved evaluation metrics at: {run.eval_file(eval_name)}")

NameError: name 'make_ppo_cfg' is not defined

## 1.5 Controlled experiment

⚠️ Start this section only **after** having a working baseline you are satisfied with and will **not tune further anymore**!

Now test one design choice of your own that is different from your baseline (e.g., a potential improvement).
A controlled experiment changes **exactly one factor** relative to your baseline and keeps everything else fixed: training budget, evaluation commands and analysis. 
Possible types of experiments:

| Type | What changes | Example |
| --- | --- | --- |
| Ablation | Remove a component | Remove an observation term or a penalty |
| Addition | Add a component | Add a reward term or observation noise |
| Substitution | Replace a design choice | A different action space or command distribution |
| Sensitivity | One parameter at a few levels | Action scale, a reward weight, `gamma` |

Results vary between seeds, so one run per configuration can mislead! 
If time allows, train the baseline and the variant with several seeds each.

### 1.5a Experiment design

**[ANSWER]** Fill this in **before** training your new variant:

- **Question:** what do you want to find out?
- **Type:** ablation, addition, substitution or sensitivity.
- **Change:** exactly what differs from the baseline.
- **Held fixed:** what stays the same.
- **Possible confounders:** what else might change as a side effect (e.g. a new action scale also changes the size of the action-rate penalty), and how you deal with it.
- **Prediction:** which metrics you expect to change, in which direction, and why.

> *Answer here, ~1-2 sentences per question*

### 1.5b Building variant config

Start from a copy of your baseline config and change exactly one thing. 
Change **either** the env config or rl config (not both) depending on your question.
Generally, env config changes lead to more interesting questions.

In [15]:
VARIANT_NAME = "no_priv_critic"
TRAIN_SEEDS = [47, 48, 49, 50, 51] 
# TO ADD 5 SEEDS PER ARM (needed for the linear-error test, see 2.2 in notebook 02):
#   1. set TRAIN_SEEDS = [47, 48, 49, 50, 51] and run this cell
#   2. run 1.5d (baseline loop, variant loop, loading cell) and 1.5f; after a kernel restart run 1.4b first,
#      1.5f needs `commands` and `eval_name` from it
#   3. copy the 10 new timestamps from the 1.5d table into SEEDS in notebook 02 (2.2c)
#   4. set TRAIN_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51] to record what was trained,
#      without re-running 1.5d / 1.5f (that would retrain and re-evaluate all 20 runs)
# Train on the same machine as the first runs and change nothing else, so the new seeds are comparable.
PRIVILEGED = ["base_lin_vel", "foot_contact", "foot_air_time", "foot_contact_forces", "joint_torques", "foot_vel"]

def make_variant_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    variant_cfg = copy.deepcopy(make_env_cfg(play=play, seed=seed))
    critic_terms = variant_cfg.observations["critic"].terms
    for name in PRIVILEGED:
        critic_terms.pop(name)
    assert critic_terms.keys() == variant_cfg.observations["actor"].terms.keys()
    return variant_cfg

def make_variant_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return make_rl_cfg(seed=seed)  # unchanged: the experiment is an env config change

### 1.5c Visualize your variant

Check that the variant builds and behaves as intended before spending GPU time on it:

In [16]:
env_cfg = make_variant_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

Setting seed: 42
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.2
   Devices:
     "cpu"      : "CPU"
     "cuda:0"   : "NVIDIA GeForce RTX 5060 Laptop GPU" (8 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/carolina-araujo/.cache/warp/1.18.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 1.11 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 0.59 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_9914afce 9914afc load on device 'cuda:0' took 0.42 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 0.22 ms  (cached)
Module _friction_dof__locals__kernel_35a89db9 35a89db load on device 'cuda:0' took 0.63 ms  (cached)
Module _limit_slide_hinge__locals__kernel_6395e840 6395e84 load on device 'cuda:0' took 0.38 ms  (cached)
Module _efc_contact_init__locals__kernel_81f4db55 81f4db5 load on device 'cuda:0' took 0.37 ms  (cached)
Module _efc_contact_j

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: http://localhost:8080

Ctrl+C received. Shutting down viewer...


(viser) Server stopped

Viewer closed!


/home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/.venv/lib/python3.12/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


### 1.5d Train your variant

Same experiment as the baseline, with your variant name. 
Then load the variant's latest run.

In [19]:
# Trains only the seeds in TRAIN_SEEDS (1.5b): with [47, ..., 51] it adds 5 new baseline runs.
for seed in TRAIN_SEEDS:
    train.train(
        env_cfg=make_env_cfg(seed=seed),
        rl_cfg=make_rl_cfg(seed=seed),
        experiment="flat",
        variant=f"baseline_seed{seed}",
    )

[train] run dir: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_09-18-28_baseline_seed47
[train] device: cuda:0, envs: 4096, seed: 47
Setting seed: 47

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 47     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 activ

In [20]:
# Trains only the seeds in TRAIN_SEEDS (1.5b): with [47, ..., 51] it adds 5 new variant runs.
for seed in TRAIN_SEEDS:
    train.train(
        env_cfg=make_variant_env_cfg(seed=seed),
        rl_cfg=make_variant_rl_cfg(seed=seed),
        experiment="flat",
        variant=f"{VARIANT_NAME}_seed{seed}",
    )

[train] run dir: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_09-45-43_no_priv_critic_seed47
[train] device: cuda:0, envs: 4096, seed: 47
Setting seed: 47

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 47     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1

In [22]:
# Latest run of each (config, seed) in TRAIN_SEEDS; the printed table gives the timestamps for SEEDS in notebook 02.
experiment_runs = {
    (name, seed): registry.runs(experiment="flat", variant=f"{name}_seed{seed}")[-1]
    for name in ("baseline", VARIANT_NAME)
    for seed in TRAIN_SEEDS
}
variant_run = experiment_runs[(VARIANT_NAME, TRAIN_SEEDS[0])]  # watched in 1.5e
registry.show(list(experiment_runs.values()))

| experiment | variant | created |
| --- | --- | --- |
| flat | baseline_seed47 | 2026-11-25_09-18-28 |
| flat | baseline_seed48 | 2026-11-25_09-22-59 |
| flat | baseline_seed49 | 2026-11-25_09-28-05 |
| flat | baseline_seed50 | 2026-11-25_09-33-09 |
| flat | baseline_seed51 | 2026-11-25_09-38-18 |
| flat | no_priv_critic_seed47 | 2026-11-25_09-45-43 |
| flat | no_priv_critic_seed48 | 2026-11-25_09-50-21 |
| flat | no_priv_critic_seed49 | 2026-11-25_09-55-05 |
| flat | no_priv_critic_seed50 | 2026-11-25_09-59-53 |
| flat | no_priv_critic_seed51 | 2026-11-25_10-04-41 |

### 1.5e Test your variant policy

Compare qualitatively with what you saw for the baseline in 1.4a. 
The quantitative comparison will take place in notebook 02.

In [ ]:
play.interactive(
    env_cfg=make_variant_env_cfg(play=True),
    rl_cfg=make_variant_rl_cfg(),
    policy=variant_run.checkpoint,
)

NameError: name 'make_variant_env_cfg' is not defined

### 1.5f Collect variant rollouts

Use exactly the same `commands` and `eval_name` as for the baseline (1.4b), so that notebook 02 compares them equally.

In [23]:
# Evaluates only the runs in experiment_runs (the TRAIN_SEEDS seeds), with the same commands and eval seed 42 as before.
for (name, seed), seed_run in experiment_runs.items():
    make_cfg = make_variant_env_cfg if name == VARIANT_NAME else make_env_cfg
    evaluate.collect_rollouts(
        env_cfg=make_cfg(play=True),
        rl_cfg=make_rl_cfg(),
        checkpoint=seed_run.checkpoint,
        commands=commands,
        name=eval_name,
        robots_per_command=10,
        duration_s=10.0,
        seed=42,
    )
    print(f"Saved evaluation metrics at: {seed_run.eval_file(eval_name)}")

[evaluate] creating 36 commands x 10 robots = 360 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 360    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----

100%|██████████| 500/500 [00:02<00:00, 183.52it/s]


Saved evaluation metrics at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_09-18-28_baseline_seed47/eval/command_grid_model_499.npz
[evaluate] creating 36 commands x 10 robots = 360 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 360    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+------

100%|██████████| 500/500 [00:02<00:00, 191.49it/s]


Saved evaluation metrics at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_09-22-59_baseline_seed48/eval/command_grid_model_499.npz
[evaluate] creating 36 commands x 10 robots = 360 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 360    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+------

100%|██████████| 500/500 [00:02<00:00, 195.35it/s]


... (11 outputs left out)

100%|██████████| 500/500 [00:02<00:00, 204.97it/s]


Saved evaluation metrics at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_09-59-53_no_priv_critic_seed50/eval/command_grid_model_499.npz
[evaluate] creating 36 commands x 10 robots = 360 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 360    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+

100%|██████████| 500/500 [00:02<00:00, 204.26it/s]


Saved evaluation metrics at: /home/carolina-araujo/Documents/EPFL/Legged_Robots/GO2-Locomotion-RL/logs/flat/2026-11-25_10-04-41_no_priv_critic_seed51/eval/command_grid_model_499.npz


## Next step

You now have a baseline and a variant, each with training logs and evaluation rollouts.
Continue in `02_flat_analysis.ipynb`.